In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df= pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df.isnull().sum()

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] =  df[num_cols].fillna(df[num_cols].mean())

In [ ]:
df.isnull().sum() == 0


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)


In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
categorical_cols #there is no categorical so no encoding needed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop('Target') #drop the target before scaling
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [ ]:
# Task 5: Write your code here:
df['Target'].hist() #the data is unbalanced

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split, KFold

x = df[features]
y = df['Target']

# TODO: Split data with stratification (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(x,y,
             test_size = 0.2 ,
             stratify= y ,
             shuffle= True  )

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4)
kf = KFold(n_splits=5, shuffle=True, random_state=42 )
result = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(x)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = x.iloc[train_index], x.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

   # Predict
  y_pred = model.predict(X_test)

  f1 = f1_score(y_test, y_pred, average='macro')
  result.append(f1)

In [ ]:
import numpy as np

print(f'avrage f1: {np.mean(f1)}')

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')

plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(f'the golden feature is { importance['feature'][0]}')

In [ ]:
# Task Bonus: Write your code here: